# 📊 Ficheros CSV en Python
*(lectura y escritura — versión explicada paso a paso)*

---

**CSV** (*Comma-Separated Values*) es el formato universal para datos tabulares: filas y columnas separadas por comas. Lo entienden Excel, LibreOffice, R, MATLAB, casi cualquier BBDD... y por supuesto Python, gracias al módulo `csv` que viene de serie.

En este notebook trabajaremos con **`notas_clase.csv`**, un fichero de ejemplo con la siguiente estructura:

```csv
nombre,curso,nota
Ana Ruiz,1,8.5
Luis García,1,6.7
Marta Ortega,2,9.2
...
```

## 1. Leer un CSV con `DictReader`

El módulo `csv` ofrece dos formas de leer:

* **`csv.reader`**: devuelve cada fila como una **lista** de cadenas. Accedes con `fila[0]`, `fila[1]`...
* **`csv.DictReader`**: devuelve cada fila como un **diccionario**, usando la **primera línea** del CSV como cabeceras. Accedes con `fila['nombre']`, `fila['nota']`... 🌟 Es mucho más legible.

Usaremos `DictReader` casi siempre.

In [ ]:
import csv

with open('notas_clase.csv', 'r') as f:
    lector = csv.DictReader(f)
    for fila in lector:
        print(fila)

Fíjate en un detalle CRUCIAL: **todos los valores del CSV son cadenas**. Aunque en el fichero pongas `8.5`, `csv` te lo entrega como `'8.5'` (string).

Si necesitas trabajar con el número, hay que **convertir** con `int()` o `float()`:

In [ ]:
with open('notas_clase.csv', 'r') as f:
    lector = csv.DictReader(f)
    for fila in lector:
        nombre = fila['nombre']
        curso  = int(fila['curso'])       # ← conversión a int
        nota   = float(fila['nota'])       # ← conversión a float
        print(f"{nombre:20} curso={curso}  nota={nota:.1f}")

## 2. Cálculos sobre datos leídos

Una vez tenemos los datos como números, aplicamos todo lo que sabemos del Tema 6: acumular en listas, calcular medias, filtrar por condición...

In [ ]:
todas_las_notas = []
aprobados = 0

with open('notas_clase.csv', 'r') as f:
    for fila in csv.DictReader(f):
        nota = float(fila['nota'])
        todas_las_notas.append(nota)
        if nota >= 5:
            aprobados = aprobados + 1

print(f'Total estudiantes: {len(todas_las_notas)}')
print(f'Aprobados:         {aprobados}')
print(f'Nota media:        {sum(todas_las_notas) / len(todas_las_notas):.2f}')
print(f'Nota máxima:       {max(todas_las_notas)}')
print(f'Nota mínima:       {min(todas_las_notas)}')

## 3. Escribir un CSV con `DictWriter`

Para escribir usamos `csv.DictWriter`. Necesitas:

1. Los **nombres de las columnas** (parámetro `fieldnames`).
2. Llamar a **`writeheader()`** para escribir la línea con los nombres.
3. Llamar a **`writerow(dict)`** para cada fila (o `writerows(lista)` para todas de golpe).

> ⚠️ **Detalle importantísimo**: al abrir para escribir CSV, pasa `newline=""` al `open()`. Sin él, en Windows aparecen líneas en blanco entre cada fila.

In [ ]:
aprobados = []

# 1) Leer y filtrar
with open('notas_clase.csv', 'r') as f:
    for fila in csv.DictReader(f):
        if float(fila['nota']) >= 5:
            aprobados.append(fila)

# 2) Escribir el resultado
with open('aprobados.csv', 'w', newline='') as f:
    campos = ['nombre', 'curso', 'nota']
    escritor = csv.DictWriter(f, fieldnames=campos)
    escritor.writeheader()
    for fila in aprobados:
        escritor.writerow(fila)

print(f'✅ {len(aprobados)} aprobados guardados en aprobados.csv')

# Verificamos
with open('aprobados.csv') as f:
    print(f.read())

## 4. Trampa clásica: el separador en español

En español usamos la **coma como decimal** (`3,14`), pero en CSV la coma es también el **separador de columnas**. Al abrir en Excel en español, `3,14` puede interpretarse como dos columnas.

La solución habitual es usar **punto y coma (`;`)** como separador cuando el CSV va a ir a Excel en configuración española. En Python se indica así:

```python
csv.DictReader(f, delimiter=';')
csv.DictWriter(f, fieldnames=..., delimiter=';')
```

En Python, los decimales SIEMPRE son con **punto** (`3.14`), aunque el idioma sea español.

## 5. Errores de codificación

Si abres un CSV creado en Excel en español (típicamente con tildes y ñ) y ves errores como `UnicodeDecodeError`, prueba a especificar la codificación:

```python
with open('datos.csv', 'r', encoding='latin-1') as f:  # o 'cp1252'
    ...
```

Si el CSV lo creaste tú con Python, se guarda por defecto en UTF-8 y todo funciona sin más.

## 🎯 Resumen: patrones de CSV

### Leer un CSV completo
```python
with open('datos.csv') as f:
    for fila in csv.DictReader(f):
        # fila es un dict
        print(fila['columna'])
```

### Escribir un CSV desde cero
```python
with open('salida.csv', 'w', newline='') as f:
    escritor = csv.DictWriter(f, fieldnames=['col1', 'col2'])
    escritor.writeheader()
    escritor.writerow({'col1': 'a', 'col2': 'b'})
```

### Filtrar un CSV
```python
with open('datos.csv') as fin, open('filtrado.csv', 'w', newline='') as fout:
    lector = csv.DictReader(fin)
    escritor = csv.DictWriter(fout, fieldnames=lector.fieldnames)
    escritor.writeheader()
    for fila in lector:
        if float(fila['nota']) >= 5:
            escritor.writerow(fila)
```